# Hand-Bill Detector — Train on Colab (free GPU)

Train the `hand_bill` detection model with **Ultralytics YOLO** on a free Google Colab GPU.

**Before starting:** have the file `hand_bill_colab.zip` ready (created locally at
`C:\Users\MY PC\Documents\git repo\export\hand_bill_colab.zip` — it contains the full dataset:
65 images with ~186 auto-generated `hand_bill` boxes).

> **Review the boxes first.** Open `bill_hand_dataset\previews\index.html` and skim the green
> boxes. Delete any bad boxes (coins, background clutter) — clean labels matter more than many
> labels. If you fixed labels in Roboflow, re-export them and run `import_roboflow.py` first.

**Steps:**
1. Run Cell 1 (installs packages)
2. Run Cell 2 (upload the zip — a file picker opens)
3. Run Cell 3 (extract + verify the dataset + stats)
4. Run Cell 4 (train — takes ~30-60 min on a T4)
5. Run Cell 5 (download the trained `best.pt` weights)
6. Run Cell 6 (optional) - upload test images or a video to preview the trained model's results inline

In [ ]:
# Cell 1 - Install packages (run once)
!pip install -q ultralytics
from ultralytics import YOLO
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - using CPU (slow)')
print('ultralytics OK')

In [ ]:
# Cell 2 - Upload the dataset zip (a file picker will open)
from google.colab import files
uploaded = files.upload()
if not uploaded:
    print('No file selected! Please run this cell again and choose hand_bill_colab.zip.')
else:
    zip_name = list(uploaded.keys())[0]
    print('Uploaded:', zip_name)

In [ ]:
# Cell 3 - Extract and verify the dataset
# If no .zip is found, this cell opens the file picker for you automatically.
import os, glob, zipfile, shutil
from google.colab import files

print('cwd:', os.getcwd())

zips = sorted(glob.glob('/content/*.zip'), key=os.path.getmtime, reverse=True)

# Remove any zero-byte zip left behind by a failed upload
zips = [z for z in zips if os.path.getsize(z) > 0]

if not zips:
    print('No .zip in /content - opening the file picker...')
    print('-> Choose hand_bill_colab.zip, then click "Upload".')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file was selected - run this cell again and pick hand_bill_colab.zip.')
    for name, data in uploaded.items():
        dst = f'/content/{os.path.basename(name)}'
        with open(dst, 'wb') as f:
            f.write(data)
        print('Saved:', dst)
    zips = sorted(glob.glob('/content/*.zip'), key=os.path.getmtime, reverse=True)

zip_path = zips[0]
print('Using:', zip_path)

if os.path.exists('/content/yolo'):
    shutil.rmtree('/content/yolo')

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content')

for split in ('train', 'val'):
    n_img = len(os.listdir(f'/content/yolo/images/{split}'))
    n_lab = len(os.listdir(f'/content/yolo/labels/{split}'))
    print(f'{split}: {n_img} images, {n_lab} labels')
print('data.yaml:')
print(open('/content/yolo/data.yaml').read())

# ---- Dataset stats (labeled images + total boxes) ----
n_img = n_lab = n_box = 0
for split in ('train', 'val'):
    n_img += len(glob.glob(f'/content/yolo/images/{split}/*.jpg'))
    for lab in glob.glob(f'/content/yolo/labels/{split}/*.txt'):
        n_lab += 1
        n_box += sum(1 for ln in open(lab) if ln.strip())
print(f'Dataset: {n_img} images, {n_lab} labeled, {n_box} total boxes')

In [ ]:
# Cell 4 - Train the model (T4 GPU, ~30-60 min for 150 epochs)
import os
os.environ['WANDB_MODE'] = 'offline'  # avoid wandb prompt

model = YOLO('yolo11n.pt')  # downloads pretrained weights on first run
results = model.train(
    data='/content/yolo/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,            # T4 handles 16 @ 640; use batch=-1 for auto-select
    patience=30,         # stop early if val mAP stops improving
    cos_lr=True,         # cosine LR schedule -> smoother convergence
    close_mosaic=10,     # disable mosaic in the last 10 epochs (small dataset)
    seed=42,
    project='/content/runs',
    name='hand_bill',
    exist_ok=True,
    verbose=True,
)
print('TRAIN_DONE')
print('Best weights:', results.save_dir / 'weights' / 'best.pt')
rd = results.results_dict
print(f"Best mAP50: {rd.get('metrics/mAP50(B)', float('nan')):.3f}   "
      f"mAP50-95: {rd.get('metrics/mAP50-95(B)', float('nan')):.3f}")

In [ ]:
# Cell 5 - Download the trained weights (best.pt) to your computer
from google.colab import files
best = '/content/runs/hand_bill/weights/best.pt'
print('Downloading', best)
files.download(best)
# Save it as:  C:\Users\MY PC\Documents\git repo\bill_hand_dataset\models\receipt_in_hand.pt

In [ ]:
# Cell 6 - Test the trained model: upload images and/or a video, preview annotated results inline
import os
from pathlib import Path

import cv2
from google.colab import files
from IPython.display import Video as IPVideo
from matplotlib import pyplot as plt
from ultralytics import YOLO

best = '/content/runs/hand_bill/weights/best.pt'
model = YOLO(best)

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
VID_EXT = {'.mp4', '.avi', '.mov', '.mkv', '.webm', '.m4v', '.ts', '.mpeg', '.mpg'}

print('Pick test images and/or a video, then click Upload:')
uploaded = files.upload()
print(f'Uploaded {len(uploaded)} file(s)')

for name, data in uploaded.items():
    ext = Path(name).suffix.lower()
    with open(name, 'wb') as f:
        f.write(data)

    if ext in IMG_EXT:
        res = model.predict(name, conf=0.25, verbose=False)[0]
        rgb = res.plot()[:, :, ::-1]  # BGR -> RGB
        plt.figure(figsize=(9, 9))
        plt.imshow(rgb)
        plt.axis('off')
        plt.title(f'{name}  -  {len(res.boxes)} detection(s)', fontsize=12)
        plt.show()

    elif ext in VID_EXT:
        cap = cv2.VideoCapture(name)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 1280
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 720
        cap.release()

        out_name = f'annotated_{Path(name).stem}.mp4'
        writer = cv2.VideoWriter(out_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
        for res in model.predict(name, stream=True, conf=0.25, verbose=False):
            writer.write(res.plot())
        writer.release()

        print(f'Annotated video saved as: {out_name}')
        try:
            display(IPVideo(out_name, embed=True))
        except Exception:
            pass
        files.download(out_name)

print('Done - download any annotated video(s) above.')

# Tip: to run the model on your webcam/phone later:
#   from ultralytics import YOLO
#   YOLO('receipt_in_hand.pt').predict(source=0, show=True)